# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [34]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
197,0.90,563.5,318.5,122.5,7.0,0.10,30.980
61,0.82,612.5,318.5,147.0,7.0,0.10,24.470
430,0.62,808.5,367.5,220.5,3.5,0.25,14.420
293,0.90,563.5,318.5,122.5,7.0,0.25,33.325
143,0.62,808.5,367.5,220.5,3.5,0.10,13.640


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

> CEVABINIZI BURAYA YAZIN
# MSE ile modelin ne kadar kotu oldugunu aciklarim.

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [35]:
# YOUR CODE HERE
from sklearn.preprocessing import StandardScaler

X = data.loc[:, 'Relative Compactness':'Glazing Area']
scaler = StandardScaler().fit(X)

X_scaled = scaler.transform(X)

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [40]:
# YOUR CODE HERE
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.linear_model import SGDRegressor

sgd_model = SGDRegressor(loss="squared_error")
sgd_model_cv = cross_validate(
    sgd_model, 
    X_scaled, 
    data['Average Temperature'],
    cv = 10, 
    scoring = ['r2','max_error']
)

sgd_model_cv

{'fit_time': array([0.00441909, 0.00214624, 0.00189209, 0.00204873, 0.00202012,
        0.00204587, 0.00164199, 0.00188398, 0.00182509, 0.00146508]),
 'score_time': array([0.00209689, 0.00051785, 0.00044012, 0.00075102, 0.00043297,
        0.00045896, 0.00036883, 0.00035691, 0.00035   , 0.00032997]),
 'test_r2': array([0.7857941 , 0.90798973, 0.89464404, 0.88366199, 0.93127897,
        0.89648497, 0.92704772, 0.91633564, 0.89538771, 0.93989585]),
 'test_max_error': array([-9.80150131, -8.65590442, -8.82689611, -9.18264347, -8.86381766,
        -8.58248286, -8.58529641, -8.75516328, -8.37667325, -7.71762681])}

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [41]:
# YOUR CODE HERE
r2 = sgd_model_cv["test_r2"].mean()
max_error_celsius = abs(sgd_model_cv["test_max_error"].max())
print(f"r2 squared_error  --> {r2_mae}  \n max error squared_error --> {max_error_mae}")

r2 squared_error  --> 0.8758542043827615  
 max error squared_error --> 10.115366221770493


### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [44]:
# YOUR CODE HERE
# SGDRegressor with Absolute Error epsilon=0 icindir
mae_model = SGDRegressor(loss="epsilon_insensitive", epsilon = 0)

# Cross Validate Model
mae_sgd = cross_validate(
    mae_model, 
    X_scaled, 
    data['Average Temperature'], 
    cv = 10,  
    scoring = ['r2','max_error']
)

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [45]:
# YOUR CODE HERE
r2_mae = mae_sgd["test_r2"].mean()
max_error_mae = abs(mae_sgd["test_max_error"].max())
print(f"r2 mae --> {r2_mae}  \n max error mae --> {max_error_mae}")

r2 mae --> 0.8764150021813375  
 max error mae --> 10.1766395782425


## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

> CEVABINIZI BURAYA YAZIN
#r2 skorları yaklaşık olarak benzerdir.
#mae (epsilon_insensitive, epsilon=0) hata orani, squared_error sgd ile loss hesaplanmistan daha yuksek hata yapma olasiligi var.

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())